In [ ]:
objname = "yasone3"
filtname = "i"

In [ ]:
from astropy.table import Table, join

import numpy as np
import arya
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

In [ ]:
import sys

In [ ]:
def load_all_catalogues(objname, filtname):
    cats = []
    imgnames = []
    for path in Path(f"../{objname}/").glob(f"img_{filtname}_*"):
        cat = Table.read(path / "forced_ap_phot.fits")
        cats.append(cat)
        imgnames.append(path.stem)

    return cats, imgnames

In [ ]:
def load_ref_cat(obj, filt):
    cat = Table.read(f"../{obj}/coadd_median_{filt}/detection.cat", hdu=2)
    cat["index"] = np.arange(len(cat))

    return cat

In [ ]:
def combine_all_catalogues(cat_ref, cats, catnames):
    cat_combined = cat_ref.copy()

    for i in range(0, len(cats)):
        cat = cats[i].copy()
        cat.rename_columns(cat.colnames, [c + "_" + catnames[i] for c in cat.colnames])

        cat_combined = join(cat_combined, cat, join_type="left", keys_left="index", keys_right="index_" + catnames[i])

    return cat_combined

In [ ]:
cats, catnames = load_all_catalogues(objname, filtname)

In [ ]:
cat_ref = load_ref_cat(objname, filtname)

In [ ]:
cat_all = combine_all_catalogues(cat_ref, cats, catnames)

In [ ]:
def get_columns(cat_all, catnames, col):
    return cat_all[[col + "_" + name for name in catnames]]

In [ ]:
ap_colnames = [col for col in cats[0].colnames if col.startswith("aperture")]

In [ ]:
tab = get_columns(cat_all, catnames, ap_colnames[0])

In [ ]:
tab

In [ ]:
from astropy.stats import mad_std

In [ ]:
cat_reduced = cat_ref.copy()

for col in ap_colnames:
    data = get_columns(cat_all, catnames, col).to_pandas()

    
    cat_reduced[col + "_median"] = np.nanmedian(data, axis=1)
    cat_reduced[col + "_err"] = mad_std(data, axis=1)
    cat_reduced[col + "_std"] = np.nanstd(data, axis=1)

In [ ]:
from scipy.stats import chi2

In [ ]:
df = 14
plt.hist(chi2(df).cdf(cat_reduced["aperture_sum_3_std"]**2 / cat_reduced["aperture_sum_err_3_median"]**2 * df), bins=np.linspace(0, 1, 100), density=True)
plt.yscale('log')

In [ ]:
plt.hist(cat_reduced["aperture_sum_3_std"]**2 / cat_reduced["aperture_sum_err_3_median"]**2, bins=np.linspace(0, 10, 100), density=True)


df = 5
f = chi2(df)

x = np.linspace(0, 10, 1000)
plt.plot(x, df*f.pdf(x*df))

In [ ]:
plt.scatter(cat_reduced["MAG_APER"][:, 3], cat_reduced["aperture_sum_3_err"] / cat_reduced["aperture_sum_err_3_median"], s=1, lw=0)
plt.ylim(-1, 6)
plt.axhline(1)
plt.xlim(-15, -5)

In [ ]:
plt.scatter(cat_reduced["MAG_APER"][:, 3], cat_reduced["aperture_sum_3_median"] /cat_reduced["FLUX_APER"][:, 3], s=1, lw=0)
plt.axhline(1)
plt.xlim(-15, -5)
plt.yscale('log')
# plt.scatter(cat_reduced["MAG_APER"][:, 3], 1+cat_reduced["aperture_sum_3_err"] /cat_reduced["aperture_sum_3_median"], s=1, lw=0, color="k", alpha=0.05)
# plt.scatter(cat_reduced["MAG_APER"][:, 3], 1-cat_reduced["aperture_sum_3_err"] /cat_reduced["aperture_sum_3_median"], s=1, lw=0, color="k", alpha=0.05)


In [ ]:
plt.scatter(cat_reduced["MAG_APER"][:, 3], cat_reduced["aperture_sum_3_median"] /cat_reduced["FLUX_APER"][:, 3], s=1, lw=0)
plt.axhline(1)
plt.xlim(-15, -5)
plt.yscale('log')
# plt.scatter(cat_reduced["MAG_APER"][:, 3], 1+cat_reduced["aperture_sum_3_err"] /cat_reduced["aperture_sum_3_median"], s=1, lw=0, color="k", alpha=0.05)
# plt.scatter(cat_reduced["MAG_APER"][:, 3], 1-cat_reduced["aperture_sum_3_err"] /cat_reduced["aperture_sum_3_median"], s=1, lw=0, color="k", alpha=0.05)


In [ ]:
N = len(catnames)

In [ ]:
for i in range(6):
    cat_reduced[f"MED_MAG_APER_{i}"] = -2.5*np.log10(cat_reduced[f"aperture_sum_{i}_median"])

    cat_reduced[f"MED_MAG_APER_{i}_ERR"] = 2.5/np.log(10) * (
        cat_reduced[f"aperture_sum_{i}_err"] / cat_reduced[f"aperture_sum_{i}_median"]
    ) * np.sqrt(np.pi/2) / np.sqrt(N-1)


In [ ]:
plt.scatter(cat_reduced["MED_MAG_APER_3"], cat_reduced["MED_MAG_APER_3_ERR"], s=1)
plt.scatter(cat_reduced["MAG_APER"][:, 3], cat_reduced["MAGERR_APER"][:, 3], s=1, alpha=0.3)
plt.ylim(0.001, 1)
plt.xlim(-20, -5)
plt.yscale("log")
